# Time-series disturbance detection

Runnable walk-through of the `disturb` pipeline. The pure-numpy
core (decompose + detect) needs only numpy; the cube build requires the
full geospatial stack (`pixi install`).

Sequence: simulate a pixel series, decompose into trend/seasonal/residual,
detect the breakpoint, then sketch the validation step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from disturb.decompose import harmonic_decompose
from disturb.detect import detect_breakpoint, detect_breakpoints_binseg, recovery_time
from disturb.trend import theil_sen_slope, mann_kendall
from disturb.demo import run_demo

## 0. One-call reproducible demo

`run_demo(0)` synthesises a small seeded NDVI-like series with a *planted*
step drop, drives the real core (`harmonic_decompose` -> `detect_breakpoint`
on the residual), writes `outputs/{series,components,summary}.{csv,json}`, and
returns the headline metrics. This is exactly what `pixi run demo` runs.

In [ ]:
metrics = run_demo(0)
metrics
# seed=0 -> n_obs=115, planted index 71, detected index 71,
# magnitude ~ -0.099, score ~ 2.70, seasonal amplitude ~ 0.226.

## 1. A synthetic disturbed pixel

Trend + annual cycle + noise, with a fire-like NDVI drop injected partway
through. (For real data, this series comes from `disturb.cube.build_ndvi_cube`.)

In [ ]:
rng = np.random.default_rng(42)
period = 365.25
t = np.arange(0, 5 * 365, 16, dtype=float)
times = np.datetime64('2018-01-01') + t.astype('timedelta64[D]')
break_at = int(t.size * 0.62)

seasonal = 0.25 * np.sin(2 * np.pi * t / period)
ndvi = 0.6 + seasonal + rng.normal(0, 0.02, t.size)
ndvi[break_at + 1:] -= 0.35  # disturbance

plt.figure(figsize=(9, 3))
plt.plot(times, ndvi, '.', ms=4)
plt.title('Synthetic NDVI series with injected disturbance')
plt.ylabel('NDVI'); plt.tight_layout()

## 2. Seasonal-trend decomposition

In [ ]:
fit = harmonic_decompose(ndvi, t=t, period=period, n_harmonics=2)
print('seasonal amplitude:', round(fit.seasonal_amplitude(1), 3))

fig, ax = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
ax[0].plot(times, fit.trend); ax[0].set_ylabel('trend')
ax[1].plot(times, fit.seasonal); ax[1].set_ylabel('seasonal')
ax[2].plot(times, fit.residual); ax[2].set_ylabel('residual')
plt.tight_layout()

## 3. Breakpoint detection on the residual

## 5. Robust trend: Theil-Sen + Mann-Kendall

Beyond the single break, ask whether the pre-break baseline is itself
trending. Theil-Sen gives a slope that one bad sample cannot drag around;
Mann-Kendall tests whether a monotonic trend is significant.

In [ ]:
# A near-flat baseline with a slow upward drift (same generator as the demo).
baseline = 0.60 + 0.00003 * t + rng.normal(0, 0.01, t.size)

print('Theil-Sen slope (per day):', round(theil_sen_slope(baseline, t), 6))
mk = mann_kendall(baseline)
print('Mann-Kendall:', mk.trend, '| S =', mk.s, '| p =', round(mk.p_value, 4))

## 6. Multiple changepoints: binary segmentation

`detect_breakpoints_binseg` recurses on the single-break CUSUM scan to find
several changepoints with no `ruptures` dependency. `recovery_time` reports
how many samples a series takes to return to its pre-break level.

In [ ]:
# Two planted steps: a drop then a partial recovery.
multi = rng.normal(0, 0.02, 120)
multi[40:] -= 0.4
multi[80:] += 0.3
breaks = detect_breakpoints_binseg(multi, max_breaks=3, threshold=1.0, min_segment=5)
for b in breaks:
    print(f'break @ index {b.index:>3}  magnitude {b.magnitude:+.3f}')

print('recovery samples after first break:',
      recovery_time(multi, breaks[0].index, tolerance=0.1))

plt.figure(figsize=(9, 3))
plt.plot(multi, '.', ms=4)
for b in breaks:
    plt.axvline(b.index, color='r', ls='--')
plt.title('Binary-segmentation changepoints'); plt.tight_layout()

In [ ]:
bp = detect_breakpoint(fit.residual, times=times, min_segment=5)
print('detected:', bp.detected)
print('break date:', bp.date)
print('magnitude:', round(bp.magnitude, 3))
print('score:', round(bp.score, 3))

plt.figure(figsize=(9, 3))
plt.plot(times, ndvi, '.', ms=4)
plt.axvline(bp.date, color='r', ls='--', label='detected break')
plt.legend(); plt.title('Detected disturbance'); plt.tight_layout()

## 4. Validation (sketch)

Stack the per-pixel `bp.date` / `bp.magnitude` into 2-D maps, then call
`disturb.validate.spatial_agreement(dates, magnitude, event_mask, event_date)`
to get the detection rate and false-alarm rate against a documented event
(see `config/aoi.yaml`).